# Earthquakes near oil & gas wells (Texas)

Cross-vertical starter: find oil & gas wells within 20 miles of the largest recent magnitude 3.0+ earthquake in Texas. Uses a pure-pandas haversine filter so it runs without any geospatial deps.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1.2,<0.2' matplotlib

# sondio >= 0.1.2 auto-resolves your key from (in order):
#   1. Colab Secrets — add a secret named SONDIO_API_KEY (🔑 sidebar, toggle Notebook access)
#   2. Kaggle Secrets — add a secret named SONDIO_API_KEY (Add-ons → Secrets)
#   3. SONDIO_API_KEY environment variable (Deepnote / Hex / SageMaker / local)
#   4. ~/.sondio/config (local CLI users)
# Grab a free key at https://sondio.io/developers
import sondio
print(f"sondio {sondio.__version__}")

In [ ]:
# Strongest recent Texas quake — pull the first page (ordered newest first).
quakes = sondio.earthquakes(state="TX", min_mag=3.0, days=365, limit=200, page=1)
print(f"{len(quakes)} earthquakes")
quakes.sort_values("magnitude", ascending=False).head()

In [ ]:
# Anchor on the largest quake.
focal = quakes.sort_values("magnitude", ascending=False).iloc[0]
lat0, lon0 = float(focal["latitude"]), float(focal["longitude"])
print(f"Focal quake: M{focal['magnitude']} on {focal['event_time']} at ({lat0:.3f}, {lon0:.3f})")

In [ ]:
# Pull TX wells, then filter by true distance with haversine.
import numpy as np

wells = sondio.oilgas.wells(country="US", state="TX", limit=500, all_pages=True)
wells = wells.dropna(subset=["latitude", "longitude"]).copy()
print(f"{len(wells)} TX wells")

In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

wells["miles_to_focal"] = haversine_miles(
    lat0, lon0, wells["latitude"].values, wells["longitude"].values
)
near = wells.loc[wells["miles_to_focal"] <= 20].sort_values("miles_to_focal")
print(f"{len(near)} wells within 20 miles")
near[["well_name", "operator_name", "locality", "subdivision", "miles_to_focal"]].head(15)

In [ ]:
# Matplotlib scatter — no basemap deps.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(wells["longitude"], wells["latitude"], s=4, c="#aaa", label="TX wells", alpha=0.5)
ax.scatter(near["longitude"], near["latitude"], s=18, c="#c44", label="wells < 20 mi")
ax.scatter([lon0], [lat0], s=180, marker="*", c="#222", label=f"M{focal['magnitude']} quake")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Texas wells near focal earthquake ({focal['event_time']})")
ax.legend(loc="upper right"); ax.grid(alpha=0.2)
plt.show()

## Next steps
- Widen the window with `days=365 * 3` or drop the state filter for CONUS-wide scope.
- Swap `state="TX"` for another seismic region (OK, CA).
- Cross with `sondio.us.phmsa.pipeline_incidents(state="TX")` to flag pipelines near the same quake.
- Related: [pipeline-safety-explorer](pipeline-safety-explorer.ipynb).